# MovieLens-20M: leader data infrastructure and Level 1 EDA

The assignment source is the Kaggle MovieLens 20M Dataset (https://www.kaggle.com/datasets/grouplens/movielens-20m-dataset). This notebook expects the matching local files to already exist. It owns data loading, quality checks, shared preprocessing tables, dataset scale, the PDF's Level 1 baseline exploration, and runtime feasibility. It does not answer the contributor Challenge Questions or advanced analyses assigned to Persons B, C, or D.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from load_data import load_movielens, summarize_frames
from preprocess import build_shared_tables, prepare_genome_scores, prepare_genome_tags
from metrics import data_quality_report, rating_distribution, runtime_feasibility
from plotting import plot_rating_distribution, save_figure

In [ ]:
data_dir = Path(os.environ.get('MOVIELENS_DATA_DIR', PROJECT_ROOT / 'data' / 'ml-20m'))
frames = load_movielens(data_dir)
ratings_raw = frames['ratings']
movies_raw = frames['movies']
tags_raw = frames.get('tags', pd.DataFrame(columns=['userId', 'movieId', 'tag', 'timestamp']))
print('Loaded data directory:', frames['_data_dir'])
display(summarize_frames(frames))

In [ ]:
tables = build_shared_tables(ratings_raw, movies_raw, tags_raw)
ratings = tables['ratings']
movies = tables['movies']
tags = tables['tags']
movie_stats = tables['movie_stats']
user_stats = tables['user_stats']
exploded_genres = tables['exploded_genres']
rating_genre_df = tables['rating_genre_df']
movie_tag_stats = tables['movie_tag_stats']

genome_scores = frames.get('genome-scores', pd.DataFrame())
genome_tags = frames.get('genome-tags', pd.DataFrame())
genome_scores = prepare_genome_scores(genome_scores) if not genome_scores.empty else None
genome_tags = prepare_genome_tags(genome_tags) if not genome_tags.empty else None

quality = data_quality_report(frames, ratings, movies, tags, genome_scores, genome_tags)
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'summary_tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
quality['file_summary'].to_csv(OUTPUT_DIR / 'file_summary.csv', index=False)
quality['key_integrity'].to_csv(OUTPUT_DIR / 'key_integrity.csv', index=False)
display(quality['file_summary'])
display(quality['key_integrity'])

## Shared tables and dataset scale

These tables are the handoff contract. The leader prepares their columns and join keys; Persons B, C, and D perform their assigned analyses on top of them.

In [ ]:
scale = pd.DataFrame({
    'metric': ['users', 'movies', 'ratings', 'tag applications', 'genome tags', 'rating start', 'rating end', 'tag start', 'tag end'],
    'value': [
        ratings['userId'].nunique(),
        movies['movieId'].nunique(),
        len(ratings),
        len(tags),
        genome_tags['tagId'].nunique() if genome_tags is not None else np.nan,
        ratings['rating_datetime'].min(),
        ratings['rating_datetime'].max(),
        tags['tag_datetime'].min() if not tags.empty else pd.NaT,
        tags['tag_datetime'].max() if not tags.empty else pd.NaT,
    ],
})
display(scale)
display(pd.DataFrame({name: [len(frame)] for name, frame in tables.items() if isinstance(frame, pd.DataFrame)}, index=['rows']).T)
display(movies[['movieId', 'title', 'release_year', 'release_year_parse_ok']].head())

## PDF Level 1 — basic movie and user exploration

The PDF asks the leader baseline to report movie counts per genre and basic user behavior. The top-five boxplot uses rating-record support as the explicit genre-popularity definition; multi-genre ratings are attributed to each constituent genre. This section is baseline context only, not an answer to B/C's Challenge Questions.

In [ ]:
genre_movie_counts = (
    exploded_genres.groupby('genre', as_index=False)['movieId']
    .nunique()
    .rename(columns={'movieId': 'movie_count'})
    .sort_values('movie_count', ascending=False)
    .reset_index(drop=True)
)
genre_movie_counts.to_csv(OUTPUT_DIR / 'level1_genre_movie_counts.csv', index=False)
display(genre_movie_counts.head(15))

genre_rating_support = (
    rating_genre_df.groupby('genre', as_index=False)
    .agg(rating_count=('rating', 'size'))
    .sort_values('rating_count', ascending=False)
)
top10 = genre_movie_counts.head(10).sort_values('movie_count')
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top10['genre'], top10['movie_count'], color='#9c6644')
ax.set_title('Top 10 genres by unique movie count')
ax.set_xlabel('Unique movies')
ax.set_ylabel('Genre')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
save_figure(fig, PROJECT_ROOT / 'figures' / 'level1_top_genres.png')
plt.show()

top5 = genre_rating_support.head(5)['genre'].tolist()
box_values = [rating_genre_df.loc[rating_genre_df['genre'].eq(genre), 'rating'].dropna() for genre in top5]
fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(box_values, patch_artist=True, boxprops={'facecolor': '#d9c2a6'})
ax.set_xticks(range(1, len(top5) + 1))
ax.set_xticklabels(top5, rotation=25, ha='right')
ax.set_title('Ratings for the top 5 genres by rating-record support')
ax.set_xlabel('Genre')
ax.set_ylabel('Rating')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
save_figure(fig, PROJECT_ROOT / 'figures' / 'level1_top_genre_rating_boxplot.png')
plt.show()

In [ ]:
user_basic_summary = user_stats[['rating_count', 'avg_rating']].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)
user_basic_summary.to_csv(OUTPUT_DIR / 'level1_user_basic_summary.csv')
display(user_basic_summary)
display(user_stats[['userId', 'rating_count', 'avg_rating']].head())

## Leader-owned rating baseline

Ratings are treated as numeric observations; the code does not assume that only integer values occur.

In [ ]:
rating_counts, rating_stats = rating_distribution(ratings)
rating_counts.to_csv(OUTPUT_DIR / 'rating_counts.csv', index=False)
rating_stats.to_csv(OUTPUT_DIR / 'rating_summary.csv', index=False)
display(rating_stats)
display(rating_counts)
rating_fig = plot_rating_distribution(rating_counts)
save_figure(rating_fig, PROJECT_ROOT / 'figures' / 'rating_distribution.png')
plt.show()

## Runtime feasibility

Runtime is not inferred from title length. This records whether a reliable runtime analysis is possible from the supplied files and identifiers.

In [ ]:
runtime_result = runtime_feasibility(frames.get('links'), movies)
pd.DataFrame([runtime_result]).to_csv(OUTPUT_DIR / 'runtime_feasibility.csv', index=False)
display(pd.Series(runtime_result, name='value').to_frame())

## Contributor boundary

Do not add B/C/D findings here. Persons B, C, and D must implement their own temporal, genre-interaction, tag/genome, advanced comparison, interpretations, limitations, and Challenge Question answers in notebooks/01-03.